In [5]:
import pandas as pd
import numpy as np
import pickle
import pkg_resources
import zipfile

In [42]:
def load_pickle(filename):
    """Load data from pickle file in package's lookups directory"""
    path = pkg_resources.resource_filename(__name__, f"lookups/{filename}")
    with open(path, 'rb') as f:
        return pickle.load(f)

def validate_recode_gss(df_in, col_code, col_data, recode_to_year, recode_from_year, fun, la_names, database_year):
    """
    Validate inputs for recode_gss function
    """
    assert isinstance(df_in, pd.DataFrame), "df_in must be a pandas DataFrame"
    assert isinstance(col_code, str), "col_code must be a string"
    assert isinstance(col_data, (str, list)), "col_data must be a string or list of strings"
    assert isinstance(recode_to_year, (int, float)), "recode_to_year must be numeric"
    assert isinstance(recode_from_year, (int, float)), "recode_from_year must be numeric"
    assert fun in ['sum', 'mean'], "fun must be 'sum' or 'mean'"

    assert recode_from_year >= 2008, "recode_from_year must be 2008 or later"
    assert recode_from_year <= database_year, f"recode_from_year cannot be later than the database year ({database_year})"
    assert recode_to_year >= 2008, "recode_to_year must be 2008 or later"
    assert recode_to_year <= database_year, f"recode_to_year cannot be later than the database year ({database_year})"

    if isinstance(col_data, str):
        col_data = [col_data]

    for col in col_data:
        assert col in df_in.columns, f"{col} is not in the input DataFrame"
        assert np.issubdtype(df_in[col].dtype, np.number), f"{col} must be numeric"

    assert col_code in df_in.columns, f"{col_code} is not in the input DataFrame"

    poss_name_cols = df_in.select_dtypes(include=[object]).columns
    for col in poss_name_cols:
        if df_in[col].isin(la_names).any():
            print(f"Warning: LA names detected in column '{col}'. Consider removing this column.")

def recode_gss(
    df_in,
    col_code='gss_code',
    col_data='value',
    fun='sum',
    recode_from_year=None,
    recode_to_year=None,
    aggregate_data=True,
    lad_code_changes=None,
    all_lad_codes_dates=None,
    database_year=2024
):
    """
    Recode GSS codes between different year vintages.
    Handles splits/merges and optionally aggregates data.
    """

    # Automatically load package-internal data if not passed
    if all_lad_codes_dates is None:
        all_lad_codes_dates = load_pickle("all_lad_codes_dates.pickle")
    if lad_code_changes is None:
        lad_code_changes = load_pickle("lad_code_changes.pickle")
    
    # Convert year column to integer to handle string/int mismatch
    if lad_code_changes['year'].dtype == 'object':
        lad_code_changes = lad_code_changes.copy()
        lad_code_changes['year'] = lad_code_changes['year'].astype(int)

    la_names = all_lad_codes_dates['gss_name'].unique()

    validate_recode_gss(df_in, col_code, col_data, recode_to_year, recode_from_year, fun, la_names, database_year)

    df = df_in.copy()
    df.rename(columns={col_code: 'gss_code'}, inplace=True)

    code_changes = lad_code_changes.copy()

    if recode_to_year < recode_from_year:
        code_changes['split2'] = code_changes['merge']
        code_changes['merge2'] = code_changes['split']
        code_changes.drop(['split', 'merge'], axis=1, inplace=True)
        code_changes.rename(columns={
            'split2': 'split',
            'merge2': 'merge',
            'changed_to_code': 'tmp1',
            'changed_from_code': 'changed_to_code',
            'tmp1': 'changed_from_code',
            'changed_to_name': 'tmp2',
            'changed_from_name': 'changed_to_name',
            'tmp2': 'changed_from_name'
        }, inplace=True)
        code_changes['year'] -= 1

    lookup = pd.DataFrame(columns=['changed_from_code', 'changed_to_code'])

    for year in range(recode_from_year, recode_to_year + 1):
        new_rows = code_changes[
            (code_changes['changed_from_code'].isin(df['gss_code'])) & (code_changes['year'] == year)
        ][['changed_from_code', 'changed_to_code']]
        lookup = pd.concat([lookup, new_rows], ignore_index=True)

        update_rows = code_changes[
            (code_changes['changed_from_code'].isin(lookup['changed_to_code'])) & (code_changes['year'] == year)
        ][['changed_from_code', 'changed_to_code']]
        if not update_rows.empty:
            lookup = lookup.merge(update_rows, how='left', left_on='changed_to_code', right_on='changed_from_code')
            lookup['changed_to_code'] = np.where(
                lookup['changed_to_code_y'].notna(), lookup['changed_to_code_y'], lookup['changed_to_code']
            )
            lookup.drop(columns=['changed_to_code_y', 'changed_from_code_y'], inplace=True)

    lookup.drop_duplicates(inplace=True)
    lookup = lookup[lookup['changed_from_code'] != lookup['changed_to_code']]

    if lookup['changed_from_code'].duplicated().any():
        lookup = lookup.groupby('changed_from_code')['changed_to_code'].apply(lambda x: ', '.join(x.unique())).reset_index()

    df = df.merge(lookup, how='left', left_on='gss_code', right_on='changed_from_code')
    
    # Show which GSS codes are being changed
    changed_codes = df[df['changed_to_code'].notna()]
    if not changed_codes.empty:
        unique_changes = changed_codes[['gss_code', 'changed_to_code']].drop_duplicates()
        for _, row in unique_changes.iterrows():
            print(f"Recoding GSS code: {row['gss_code']} -> {row['changed_to_code']}")
    else:
        print("No GSS codes required recoding.")
    
    df['gss_code'] = df['changed_to_code'].fillna(df['gss_code'])
    df.drop(columns=['changed_from_code', 'changed_to_code'], inplace=True)

    if isinstance(col_data, str):
        col_data = [col_data]

    group_cols = [col for col in df.columns if col not in col_data]

    if aggregate_data:
        if fun == 'sum':
            df = df.groupby(group_cols, as_index=False)[col_data].sum()
        elif fun == 'mean':
            df = df.groupby(group_cols, as_index=False)[col_data].mean()

    df.rename(columns={'gss_code': col_code}, inplace=True)
    df = df[df_in.columns]

    return df


In [43]:
LOCAL_PARQUET_ZIP_PATH = "/Users/user1/Documents/domestic_rates_preprocessing/test_data/old_series_data/origin_destination_2002_to_2020.parquet.zip"
LOCAL_NEW_SERIES_PARQ = "/Users/user1/Documents/domestic_rates_preprocessing/test_data/combined/cleaned_data_combined.parquet"
LOCAL_OUTPUT_DIR = "/Users/user1/Documents/domestic_rates_preprocessing/test_data/new_geog_combined/output_parquet"

START_YR_NEW_SERIES = 2012
GSS_OLD_YEAR = 2021
GSS_NEW_YEAR = 2023

print("🚀 Starting local combine_series...")

# 1. Unzip and load the old series Parquet
print("📥 Reading ZIP from local file...")
with zipfile.ZipFile(LOCAL_PARQUET_ZIP_PATH, 'r') as zip_ref:
    parquet_name = zip_ref.namelist()[0]
    print(f"📦 Unzipping and reading {parquet_name}")
    with zip_ref.open(parquet_name) as f:
        df_old = pd.read_parquet(f)

df_old = df_old[df_old['year'] < START_YR_NEW_SERIES]
print(f"✅ Loaded old series shape: {df_old.shape}")

# 2. Recode gss_in
df_in_gss = df_old[df_old['gss_in'].str.contains("E0|W0")]
df_not_in_gss = df_old[~df_old['gss_in'].str.contains("E0|W0")]

print(f"🔄 Recoding gss_in from {GSS_OLD_YEAR} to {GSS_NEW_YEAR}...")
df_in_gss_recoded = recode_gss(
    df_in=df_in_gss,
    col_code='gss_in',
    col_data='value',
    fun='sum',
    recode_from_year=GSS_OLD_YEAR,
    recode_to_year=GSS_NEW_YEAR,
)

🚀 Starting local combine_series...
📥 Reading ZIP from local file...
📦 Unzipping and reading origin_destination_2002_to_2020.parquet
✅ Loaded old series shape: (19705748, 6)
✅ Loaded old series shape: (19705748, 6)
🔄 Recoding gss_in from 2021 to 2023...
🔄 Recoding gss_in from 2021 to 2023...
Recoding GSS code: E07000026 -> E06000063
Recoding GSS code: E07000027 -> E06000064
Recoding GSS code: E07000028 -> E06000063
Recoding GSS code: E07000029 -> E06000063
Recoding GSS code: E07000030 -> E06000064
Recoding GSS code: E07000031 -> E06000064
Recoding GSS code: E07000163 -> E06000065
Recoding GSS code: E07000164 -> E06000065
Recoding GSS code: E07000165 -> E06000065
Recoding GSS code: E07000166 -> E06000065
Recoding GSS code: E07000167 -> E06000065
Recoding GSS code: E07000168 -> E06000065
Recoding GSS code: E07000169 -> E06000065
Recoding GSS code: E07000187 -> E06000066
Recoding GSS code: E07000188 -> E06000066
Recoding GSS code: E07000189 -> E06000066
Recoding GSS code: E07000246 -> E060

In [27]:
df_new_series = pd.read_parquet('/Users/user1/Documents/domestic_rates_preprocessing/test_data/3_clean_data_combined/cleaned_data_combined.parquet')
print(f"✅ Loaded new series shape: {df_new_series.shape}")

✅ Loaded new series shape: (1806380, 116)


In [28]:
df_new_series 

,outla,inla,sex,year,age_0,age_1,age_2,age_3,age_4,age_5,...,age_102,age_103,age_104,age_105,age_106,age_107,age_108,age_109,age_110,age_111
0,E06000001,E06000002,F,2018,0.3216,1.4373,2.9503,0.0000,0.0000,2.9367,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
1,E06000001,E06000002,M,2018,0.0000,0.0000,1.4466,4.5061,0.0000,3.0329,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
2,E06000001,E06000003,F,2018,0.0000,0.0000,0.0000,1.4404,1.5002,1.4801,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
3,E06000001,E06000003,M,2018,0.0000,0.0000,1.4519,0.0000,1.5057,1.4899,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
4,E06000001,E06000004,F,2018,3.5056,8.7275,10.3012,11.7748,5.9903,2.9396,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1806375,W06000024,W06000021,M,2020,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN
1806376,W06000024,W06000022,F,2020,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN
1806377,W06000024,W06000022,M,2020,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN
1806378,W06000024,W06000023,F,2020,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN


In [ ]:
df_old

,gss_out,gss_in,year,sex,age,value
0,E06000001,E06000002,2002.0,female,7.0,1.1805
1,E06000001,E06000002,2002.0,female,8.0,1.1805
2,E06000001,E06000002,2002.0,female,9.0,1.1805
3,E06000001,E06000002,2002.0,female,11.0,2.3610
4,E06000001,E06000002,2002.0,female,12.0,1.1805
...,...,...,...,...,...,...
29937902,W06000024,W06000023,2011.0,male,34.0,1.1264
29937903,W06000024,W06000023,2011.0,male,38.0,1.1264
29937904,W06000024,W06000023,2011.0,male,43.0,1.1264
29937905,W06000024,W06000023,2011.0,male,52.0,1.1264


In [10]:
df_in_gss_recoded 

,gss_out,gss_in,year,sex,age,value
0,E06000001,E06000002,2002.0,female,7.0,1.1805
1,E06000001,E06000002,2002.0,female,8.0,1.1805
2,E06000001,E06000002,2002.0,female,9.0,1.1805
3,E06000001,E06000002,2002.0,female,11.0,2.3610
4,E06000001,E06000002,2002.0,female,12.0,1.1805
...,...,...,...,...,...,...
19438718,W06000024,W06000023,2011.0,male,34.0,1.1264
19438719,W06000024,W06000023,2011.0,male,38.0,1.1264
19438720,W06000024,W06000023,2011.0,male,43.0,1.1264
19438721,W06000024,W06000023,2011.0,male,52.0,1.1264


## Read in 2021 and 2023 LA codes 

In [21]:
LAD_2023 = pd.read_csv('/Users/user1/Downloads/Local_Authority_Districts_(April_2023)_Names_and_Codes_in_the_United_Kingdom.csv')
LAD_2021 = pd.read_csv('/Users/user1/Downloads/Local_Authority_Districts_December_2021_GB_BUC_2022_5377558106745745141.csv')

In [22]:
#FILTER FIRST COLUMN
LAD_2023 = LAD_2023[LAD_2023.columns[0]]
LAD_2021 = LAD_2021[LAD_2021.columns[1]]

In [24]:
print("LAD_2023 shape:", LAD_2023.shape)
print("LAD_2021 shape:", LAD_2021.shape)
print("\nFirst 10 codes in LAD_2023:")
print(LAD_2023.head(10))
print("\nFirst 10 codes in LAD_2021:")
print(LAD_2021.head(10))

# Check if they have different codes
codes_2023 = set(LAD_2023.values)
codes_2021 = set(LAD_2021.values)

print(f"\nNumber of unique codes in 2023: {len(codes_2023)}")
print(f"Number of unique codes in 2021: {len(codes_2021)}")

# Find codes that are in 2021 but not in 2023
codes_only_in_2021 = codes_2021 - codes_2023
print(f"\nCodes in 2021 but NOT in 2023 ({len(codes_only_in_2021)}):")
for code in sorted(codes_only_in_2021):
    print(code)

# Find codes that are in 2023 but not in 2021
codes_only_in_2023 = codes_2023 - codes_2021
print(f"\nCodes in 2023 but NOT in 2021 ({len(codes_only_in_2023)}):")
for code in sorted(codes_only_in_2023):
    print(code)

LAD_2023 shape: (361,)
LAD_2021 shape: (363,)

First 10 codes in LAD_2023:
0    E06000001
1    E06000002
2    E06000003
3    E06000004
4    E06000005
5    E06000006
6    E06000007
7    E06000008
8    E06000009
9    E06000010
Name: LAD23CD, dtype: object

First 10 codes in LAD_2021:
0    E06000001
1    E06000002
2    E06000003
3    E06000004
4    E06000005
5    E06000006
6    E06000007
7    E06000008
8    E06000009
9    E06000010
Name: LAD21CD, dtype: object

Number of unique codes in 2023: 361
Number of unique codes in 2021: 363

Codes in 2021 but NOT in 2023 (17):
E07000026
E07000027
E07000028
E07000029
E07000030
E07000031
E07000163
E07000164
E07000165
E07000166
E07000167
E07000168
E07000169
E07000187
E07000188
E07000189
E07000246

Codes in 2023 but NOT in 2021 (15):
E06000063
E06000064
E06000065
E06000066
N09000001
N09000002
N09000003
N09000004
N09000005
N09000006
N09000007
N09000008
N09000009
N09000010
N09000011


In [25]:
# Now let's check which LAD codes are present in df_old
print("\n" + "="*60)
print("ANALYZING df_old GSS CODES")
print("="*60)

# Get unique codes from gss_in and gss_out in df_old
gss_in_codes = set(df_old['gss_in'].unique())
gss_out_codes = set(df_old['gss_out'].unique())
all_df_old_codes = gss_in_codes.union(gss_out_codes)

print(f"Unique codes in df_old gss_in: {len(gss_in_codes)}")
print(f"Unique codes in df_old gss_out: {len(gss_out_codes)}")
print(f"Total unique codes in df_old: {len(all_df_old_codes)}")

# Check overlap with 2021 and 2023 codes
codes_2023 = set(LAD_2023.values)
codes_2021 = set(LAD_2021.values)

# Check which 2021-only codes appear in df_old
codes_2021_only = codes_2021 - codes_2023
codes_2023_only = codes_2023 - codes_2021

df_old_has_2021_only = all_df_old_codes.intersection(codes_2021_only)
df_old_has_2023_only = all_df_old_codes.intersection(codes_2023_only)

print(f"\nCodes in df_old that are ONLY in LAD_2021 ({len(df_old_has_2021_only)}):")
for code in sorted(df_old_has_2021_only):
    print(f"  {code}")

print(f"\nCodes in df_old that are ONLY in LAD_2023 ({len(df_old_has_2023_only)}):")
for code in sorted(df_old_has_2023_only):
    print(f"  {code}")

# Calculate overlap percentages
overlap_2021 = len(all_df_old_codes.intersection(codes_2021))
overlap_2023 = len(all_df_old_codes.intersection(codes_2023))

print(f"\nOVERLAP ANALYSIS:")
print(f"df_old codes that match LAD_2021: {overlap_2021}/{len(all_df_old_codes)} ({overlap_2021/len(all_df_old_codes)*100:.1f}%)")
print(f"df_old codes that match LAD_2023: {overlap_2023}/{len(all_df_old_codes)} ({overlap_2023/len(all_df_old_codes)*100:.1f}%)")

# Conclusion
if len(df_old_has_2021_only) > 0 and len(df_old_has_2023_only) == 0:
    print(f"\n🔍 CONCLUSION: df_old appears to use LAD_2021 codes")
    print(f"   Evidence: Contains {len(df_old_has_2021_only)} codes that only exist in 2021")
elif len(df_old_has_2023_only) > 0 and len(df_old_has_2021_only) == 0:
    print(f"\n🔍 CONCLUSION: df_old appears to use LAD_2023 codes")
    print(f"   Evidence: Contains {len(df_old_has_2023_only)} codes that only exist in 2023")
elif len(df_old_has_2021_only) == 0 and len(df_old_has_2023_only) == 0:
    print(f"\n🔍 CONCLUSION: df_old codes are compatible with both LAD_2021 and LAD_2023")
    print(f"   Evidence: No codes unique to either vintage found")
else:
    print(f"\n🔍 CONCLUSION: df_old contains mixed vintages or other codes")
    print(f"   Evidence: Contains codes from both 2021-only and 2023-only")


ANALYZING df_old GSS CODES
Unique codes in df_old gss_in: 333
Unique codes in df_old gss_out: 333
Total unique codes in df_old: 333

Codes in df_old that are ONLY in LAD_2021 (17):
  E07000026
  E07000027
  E07000028
  E07000029
  E07000030
  E07000031
  E07000163
  E07000164
  E07000165
  E07000166
  E07000167
  E07000168
  E07000169
  E07000187
  E07000188
  E07000189
  E07000246

Codes in df_old that are ONLY in LAD_2023 (0):

OVERLAP ANALYSIS:
df_old codes that match LAD_2021: 331/333 (99.4%)
df_old codes that match LAD_2023: 314/333 (94.3%)

🔍 CONCLUSION: df_old appears to use LAD_2021 codes
   Evidence: Contains 17 codes that only exist in 2021


In [29]:
# Now let's check which LAD codes are present in df_new_series
print("\n" + "="*60)
print("ANALYZING df_new_series LAD CODES")
print("="*60)

# Get unique codes from outla and inla in df_new_series
outla_codes = set(df_new_series['outla'].unique())
inla_codes = set(df_new_series['inla'].unique())
all_df_new_codes = outla_codes.union(inla_codes)

print(f"Unique codes in df_new_series outla: {len(outla_codes)}")
print(f"Unique codes in df_new_series inla: {len(inla_codes)}")
print(f"Total unique codes in df_new_series: {len(all_df_new_codes)}")

# Check overlap with 2021 and 2023 codes
codes_2023 = set(LAD_2023.values)
codes_2021 = set(LAD_2021.values)

# Check which 2021-only codes appear in df_new_series
codes_2021_only = codes_2021 - codes_2023
codes_2023_only = codes_2023 - codes_2021

df_new_has_2021_only = all_df_new_codes.intersection(codes_2021_only)
df_new_has_2023_only = all_df_new_codes.intersection(codes_2023_only)

print(f"\nCodes in df_new_series that are ONLY in LAD_2021 ({len(df_new_has_2021_only)}):")
for code in sorted(df_new_has_2021_only):
    print(f"  {code}")

print(f"\nCodes in df_new_series that are ONLY in LAD_2023 ({len(df_new_has_2023_only)}):")
for code in sorted(df_new_has_2023_only):
    print(f"  {code}")

# Calculate overlap percentages
overlap_2021_new = len(all_df_new_codes.intersection(codes_2021))
overlap_2023_new = len(all_df_new_codes.intersection(codes_2023))

print(f"\nOVERLAP ANALYSIS:")
print(f"df_new_series codes that match LAD_2021: {overlap_2021_new}/{len(all_df_new_codes)} ({overlap_2021_new/len(all_df_new_codes)*100:.1f}%)")
print(f"df_new_series codes that match LAD_2023: {overlap_2023_new}/{len(all_df_new_codes)} ({overlap_2023_new/len(all_df_new_codes)*100:.1f}%)")

# Conclusion
if len(df_new_has_2021_only) > 0 and len(df_new_has_2023_only) == 0:
    print(f"\n🔍 CONCLUSION: df_new_series appears to use LAD_2021 codes")
    print(f"   Evidence: Contains {len(df_new_has_2021_only)} codes that only exist in 2021")
elif len(df_new_has_2023_only) > 0 and len(df_new_has_2021_only) == 0:
    print(f"\n🔍 CONCLUSION: df_new_series appears to use LAD_2023 codes")
    print(f"   Evidence: Contains {len(df_new_has_2023_only)} codes that only exist in 2023")
elif len(df_new_has_2021_only) == 0 and len(df_new_has_2023_only) == 0:
    print(f"\n🔍 CONCLUSION: df_new_series codes are compatible with both LAD_2021 and LAD_2023")
    print(f"   Evidence: No codes unique to either vintage found")
else:
    print(f"\n🔍 CONCLUSION: df_new_series contains mixed vintages or other codes")
    print(f"   Evidence: Contains codes from both 2021-only and 2023-only")


ANALYZING df_new_series LAD CODES
Unique codes in df_new_series outla: 320
Unique codes in df_new_series inla: 320
Total unique codes in df_new_series: 320

Codes in df_new_series that are ONLY in LAD_2021 (0):

Codes in df_new_series that are ONLY in LAD_2023 (4):
  E06000063
  E06000064
  E06000065
  E06000066

OVERLAP ANALYSIS:
df_new_series codes that match LAD_2021: 314/320 (98.1%)
df_new_series codes that match LAD_2023: 318/320 (99.4%)

🔍 CONCLUSION: df_new_series appears to use LAD_2023 codes
   Evidence: Contains 4 codes that only exist in 2023


In [40]:
START_YR_NEW_SERIES = 2012
GSS_OLD_YEAR = 2021
GSS_NEW_YEAR = 2023

# 2. Recode gss_in
df_in_gss = df_old[df_old['gss_in'].str.contains("E0|W0")]
df_not_in_gss = df_old[~df_old['gss_in'].str.contains("E0|W0")]

print(f"🔄 Recoding gss_in from {GSS_OLD_YEAR} to {GSS_NEW_YEAR}...")
df_in_gss_recoded = recode_gss(
    df_in=df_in_gss,
    col_code='gss_in',
    col_data='value',
    fun='sum',
    recode_from_year=GSS_OLD_YEAR,
    recode_to_year=GSS_NEW_YEAR,)

🔄 Recoding gss_in from 2021 to 2023...
No GSS codes required recoding.
No GSS codes required recoding.


In [34]:
# Let's investigate why no recoding occurred
print("\n" + "="*50)
print("DIAGNOSTIC: Why no recoding occurred?")
print("="*50)

# Load the lookup data to see what changes are available
lad_code_changes = load_pickle("lad_code_changes.pickle")
print(f"Total changes in lookup: {len(lad_code_changes)}")

# Check what years are available in the changes data
available_years = sorted(lad_code_changes['year'].unique())
print(f"Available years in changes data: {available_years}")

# Check if there are any changes for 2022 or 2023
changes_2022 = lad_code_changes[lad_code_changes['year'] == 2022]
changes_2023 = lad_code_changes[lad_code_changes['year'] == 2023]
print(f"Changes available for 2022: {len(changes_2022)}")
print(f"Changes available for 2023: {len(changes_2023)}")

if len(changes_2022) > 0:
    print("Sample 2022 changes:")
    print(changes_2022[['changed_from_code', 'changed_to_code', 'year']].head())

if len(changes_2023) > 0:
    print("Sample 2023 changes:")
    print(changes_2023[['changed_from_code', 'changed_to_code', 'year']].head())

# Check which codes from df_old are in the changes data
df_old_gss_codes = set(df_old['gss_in'].unique())
codes_in_changes = set(lad_code_changes['changed_from_code'].unique())
overlap = df_old_gss_codes.intersection(codes_in_changes)

print(f"\nCodes in df_old gss_in: {len(df_old_gss_codes)}")
print(f"Codes that can be changed (in lookup): {len(codes_in_changes)}")
print(f"Overlap (codes in df_old that could be changed): {len(overlap)}")

if len(overlap) > 0:
    print("Sample codes that could be changed:")
    for code in list(overlap)[:5]:
        print(f"  {code}")


DIAGNOSTIC: Why no recoding occurred?
Total changes in lookup: 90
Available years in changes data: ['2009', '2012', '2013', '2018', '2019', '2020', '2021', '2023']
Changes available for 2022: 0
Changes available for 2023: 0

Codes in df_old gss_in: 333
Codes that can be changed (in lookup): 90
Overlap (codes in df_old that could be changed): 19
Sample codes that could be changed:
  E07000030
  E07000164
  E07000027
  E07000026
  E07000189


In [35]:
# Let's check what changes are available for 2021
changes_2021 = lad_code_changes[lad_code_changes['year'] == 2021]
print(f"\nChanges available for 2021: {len(changes_2021)}")

if len(changes_2021) > 0:
    print("All 2021 changes:")
    print(changes_2021[['changed_from_code', 'changed_to_code', 'year']])
    
    # Check if any of these FROM codes are in our df_old data
    codes_that_change_in_2021 = set(changes_2021['changed_from_code'])
    overlap_2021 = df_old_gss_codes.intersection(codes_that_change_in_2021)
    print(f"\nCodes in df_old that change in 2021: {len(overlap_2021)}")
    for code in sorted(overlap_2021):
        change_info = changes_2021[changes_2021['changed_from_code'] == code]
        print(f"  {code} -> {change_info['changed_to_code'].iloc[0]}")

print(f"\n💡 SUGGESTION: The changes might be recorded for 2021, not 2022/2023.")
print(f"   Try recoding from 2020 to 2021 instead of 2021 to 2023.")


Changes available for 2021: 0

💡 SUGGESTION: The changes might be recorded for 2021, not 2022/2023.
   Try recoding from 2020 to 2021 instead of 2021 to 2023.


In [36]:
# Let's see what the most recent changes are
print("Changes by year:")
for year in sorted(lad_code_changes['year'].unique()):
    year_changes = lad_code_changes[lad_code_changes['year'] == year]
    print(f"  {year}: {len(year_changes)} changes")

# Let's look at the most recent changes (2023 in the available years list)
# Actually, let's check 2020 since that was the most recent with changes
changes_2020 = lad_code_changes[lad_code_changes['year'] == 2020]
if len(changes_2020) > 0:
    print(f"\n2020 changes ({len(changes_2020)}):")
    print(changes_2020[['changed_from_code', 'changed_to_code', 'year']])
    
    # Check overlap with our data
    codes_that_change_in_2020 = set(changes_2020['changed_from_code'])
    overlap_2020 = df_old_gss_codes.intersection(codes_that_change_in_2020)
    print(f"\nCodes in df_old that would change in 2020: {len(overlap_2020)}")
    if len(overlap_2020) > 0:
        for code in sorted(overlap_2020)[:10]:  # Show first 10
            change_info = changes_2020[changes_2020['changed_from_code'] == code]
            if not change_info.empty:
                print(f"  {code} -> {change_info['changed_to_code'].iloc[0]}")

print(f"\n🤔 ANALYSIS: Your data might already be in the 2023 format, or")
print(f"    the changes from 2021->2023 might not be in this lookup dataset.")
print(f"    The lookup seems to have older changes (up to 2020).")

Changes by year:
  2009: 40 changes
  2012: 2 changes
  2013: 5 changes
  2018: 1 changes
  2019: 14 changes
  2020: 4 changes
  2021: 7 changes
  2023: 17 changes

🤔 ANALYSIS: Your data might already be in the 2023 format, or
    the changes from 2021->2023 might not be in this lookup dataset.
    The lookup seems to have older changes (up to 2020).


In [37]:
# Wait! There ARE 17 changes for 2023. Let me check them
changes_2023_detailed = lad_code_changes[lad_code_changes['year'] == 2023]
print(f"2023 changes ({len(changes_2023_detailed)}):")
print(changes_2023_detailed[['changed_from_code', 'changed_to_code', 'year']])

# Check if the data type is causing issues
print(f"\nData type of year column: {lad_code_changes['year'].dtype}")
print(f"Unique values in year column: {sorted(lad_code_changes['year'].unique())}")

# Check overlap with our data
codes_that_change_in_2023 = set(changes_2023_detailed['changed_from_code'])
overlap_2023 = df_old_gss_codes.intersection(codes_that_change_in_2023)
print(f"\nCodes in df_old that would change in 2023: {len(overlap_2023)}")
if len(overlap_2023) > 0:
    print("These codes from df_old would be changed:")
    for code in sorted(overlap_2023):
        change_info = changes_2023_detailed[changes_2023_detailed['changed_from_code'] == code]
        if not change_info.empty:
            print(f"  {code} -> {change_info['changed_to_code'].iloc[0]}")

# Let's also check 2021 changes more carefully  
changes_2021_detailed = lad_code_changes[lad_code_changes['year'] == 2021]
print(f"\n2021 changes ({len(changes_2021_detailed)}):")
if len(changes_2021_detailed) > 0:
    print(changes_2021_detailed[['changed_from_code', 'changed_to_code', 'year']])

2023 changes (0):
Empty DataFrame
Columns: [changed_from_code, changed_to_code, year]
Index: []

Data type of year column: object
Unique values in year column: ['2009', '2012', '2013', '2018', '2019', '2020', '2021', '2023']

Codes in df_old that would change in 2023: 0

2021 changes (0):


In [38]:
# Debug the year filtering issue
print("Debugging year values:")
year_values = lad_code_changes['year'].value_counts()
print(year_values)

# Check for whitespace or formatting issues
print("\nActual year values (with quotes to see whitespace):")
for year in lad_code_changes['year'].unique():
    print(f"'{year}' (type: {type(year)})")

# Try different filtering approaches
print(f"\nTrying different filters for 2023:")
print(f"year == '2023': {len(lad_code_changes[lad_code_changes['year'] == '2023'])}")
print(f"year == 2023: {len(lad_code_changes[lad_code_changes['year'] == 2023])}")
print(f"year.str.contains('2023'): {len(lad_code_changes[lad_code_changes['year'].str.contains('2023', na=False)])}")

# Let's look at the actual data around where 2023 should be
sample_2023 = lad_code_changes[lad_code_changes['year'].str.contains('2023', na=False)]
if len(sample_2023) > 0:
    print(f"\nActual 2023 data:")
    print(sample_2023[['changed_from_code', 'changed_to_code', 'year']].head())
    
    # Check if any of these codes are in our df_old
    codes_in_2023_changes = set(sample_2023['changed_from_code'])
    overlap_with_df_old = df_old_gss_codes.intersection(codes_in_2023_changes)
    print(f"\nCodes from df_old that appear in 2023 changes: {len(overlap_with_df_old)}")
    for code in sorted(overlap_with_df_old)[:10]:
        print(f"  {code}")

Debugging year values:
year
2009    40
2023    17
2019    14
2021     7
2013     5
2020     4
2012     2
2018     1
Name: count, dtype: int64

Actual year values (with quotes to see whitespace):
'2009' (type: <class 'str'>)
'2012' (type: <class 'str'>)
'2013' (type: <class 'str'>)
'2018' (type: <class 'str'>)
'2019' (type: <class 'str'>)
'2020' (type: <class 'str'>)
'2021' (type: <class 'str'>)
'2023' (type: <class 'str'>)

Trying different filters for 2023:
year == '2023': 17
year == 2023: 0
year.str.contains('2023'): 17

Actual 2023 data:
       changed_from_code changed_to_code  year
499725         E07000026       E06000063  2023
499726         E07000028       E06000063  2023
499727         E07000029       E06000063  2023
499728         E07000027       E06000064  2023
499729         E07000030       E06000064  2023

Codes from df_old that appear in 2023 changes: 17
  E07000026
  E07000027
  E07000028
  E07000029
  E07000030
  E07000031
  E07000163
  E07000164
  E07000165
  E07000166


In [45]:
# The issue is that the lookup data has string years, but the function expects numeric years
# Let's fix this by converting the lookup data years to integers
print("🔧 Converting lookup data years to integers...")

# Load and convert the lookup data
lad_code_changes_fixed = load_pickle("lad_code_changes.pickle").copy()
lad_code_changes_fixed['year'] = lad_code_changes_fixed['year'].astype(int)

print(f"Year data types after conversion: {lad_code_changes_fixed['year'].dtype}")
print(f"Available years: {sorted(lad_code_changes_fixed['year'].unique())}")

# Now try the recode with the fixed lookup data
print("🔄 Trying recode with converted lookup data...")
df_in_gss_recoded_fixed = recode_gss(
    df_in=df_in_gss,
    col_code='gss_in',
    col_data='value',
    fun='sum',
    recode_from_year=2021,  # Back to integers
    recode_to_year=2023,    # Back to integers
    lad_code_changes=lad_code_changes_fixed  # Pass the fixed lookup data
)

🔧 Converting lookup data years to integers...
Year data types after conversion: int64
Available years: [np.int64(2009), np.int64(2012), np.int64(2013), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2023)]
🔄 Trying recode with converted lookup data...
Recoding GSS code: E07000026 -> E06000063
Recoding GSS code: E07000027 -> E06000064
Recoding GSS code: E07000028 -> E06000063
Recoding GSS code: E07000029 -> E06000063
Recoding GSS code: E07000030 -> E06000064
Recoding GSS code: E07000031 -> E06000064
Recoding GSS code: E07000163 -> E06000065
Recoding GSS code: E07000164 -> E06000065
Recoding GSS code: E07000165 -> E06000065
Recoding GSS code: E07000166 -> E06000065
Recoding GSS code: E07000167 -> E06000065
Recoding GSS code: E07000168 -> E06000065
Recoding GSS code: E07000169 -> E06000065
Recoding GSS code: E07000187 -> E06000066
Recoding GSS code: E07000188 -> E06000066
Recoding GSS code: E07000189 -> E06000066
Recoding GSS code: E07000246 -> E06000066


In [46]:
# Let's check which LAD vintage the recoded data uses
print("\n" + "="*60)
print("ANALYZING df_in_gss_recoded GSS CODES")
print("="*60)

# Get unique codes from gss_in in the recoded data
recoded_gss_in_codes = set(df_in_gss_recoded['gss_in'].unique())

print(f"Unique codes in df_in_gss_recoded gss_in: {len(recoded_gss_in_codes)}")

# Check overlap with 2021 and 2023 codes
codes_2023 = set(LAD_2023.values)
codes_2021 = set(LAD_2021.values)

# Check which 2021-only codes appear in recoded data
codes_2021_only = codes_2021 - codes_2023
codes_2023_only = codes_2023 - codes_2021

recoded_has_2021_only = recoded_gss_in_codes.intersection(codes_2021_only)
recoded_has_2023_only = recoded_gss_in_codes.intersection(codes_2023_only)

print(f"\nCodes in df_in_gss_recoded that are ONLY in LAD_2021 ({len(recoded_has_2021_only)}):")
for code in sorted(recoded_has_2021_only):
    print(f"  {code}")

print(f"\nCodes in df_in_gss_recoded that are ONLY in LAD_2023 ({len(recoded_has_2023_only)}):")
for code in sorted(recoded_has_2023_only):
    print(f"  {code}")

# Calculate overlap percentages
overlap_2021_recoded = len(recoded_gss_in_codes.intersection(codes_2021))
overlap_2023_recoded = len(recoded_gss_in_codes.intersection(codes_2023))

print(f"\nOVERLAP ANALYSIS:")
print(f"df_in_gss_recoded codes that match LAD_2021: {overlap_2021_recoded}/{len(recoded_gss_in_codes)} ({overlap_2021_recoded/len(recoded_gss_in_codes)*100:.1f}%)")
print(f"df_in_gss_recoded codes that match LAD_2023: {overlap_2023_recoded}/{len(recoded_gss_in_codes)} ({overlap_2023_recoded/len(recoded_gss_in_codes)*100:.1f}%)")

# Conclusion
if len(recoded_has_2021_only) > 0 and len(recoded_has_2023_only) == 0:
    print(f"\n🔍 CONCLUSION: df_in_gss_recoded appears to use LAD_2021 codes")
    print(f"   Evidence: Contains {len(recoded_has_2021_only)} codes that only exist in 2021")
elif len(recoded_has_2023_only) > 0 and len(recoded_has_2021_only) == 0:
    print(f"\n🔍 CONCLUSION: df_in_gss_recoded appears to use LAD_2023 codes")
    print(f"   Evidence: Contains {len(recoded_has_2023_only)} codes that only exist in 2023")
elif len(recoded_has_2021_only) == 0 and len(recoded_has_2023_only) == 0:
    print(f"\n🔍 CONCLUSION: df_in_gss_recoded codes are compatible with both LAD_2021 and LAD_2023")
    print(f"   Evidence: No codes unique to either vintage found")
else:
    print(f"\n🔍 CONCLUSION: df_in_gss_recoded contains mixed vintages or other codes")
    print(f"   Evidence: Contains codes from both 2021-only and 2023-only")

# Compare with original data
original_gss_in_codes = set(df_in_gss['gss_in'].unique())
print(f"\n📊 COMPARISON WITH ORIGINAL:")
print(f"Original df_in_gss codes: {len(original_gss_in_codes)}")
print(f"Recoded df_in_gss_recoded codes: {len(recoded_gss_in_codes)}")

# Check which codes changed
codes_that_disappeared = original_gss_in_codes - recoded_gss_in_codes
codes_that_appeared = recoded_gss_in_codes - original_gss_in_codes

print(f"Codes that disappeared after recoding: {len(codes_that_disappeared)}")
for code in sorted(codes_that_disappeared):
    print(f"  {code}")

print(f"Codes that appeared after recoding: {len(codes_that_appeared)}")
for code in sorted(codes_that_appeared):
    print(f"  {code}")


ANALYZING df_in_gss_recoded GSS CODES
Unique codes in df_in_gss_recoded gss_in: 318

Codes in df_in_gss_recoded that are ONLY in LAD_2021 (0):

Codes in df_in_gss_recoded that are ONLY in LAD_2023 (4):
  E06000063
  E06000064
  E06000065
  E06000066

OVERLAP ANALYSIS:
df_in_gss_recoded codes that match LAD_2021: 314/318 (98.7%)
df_in_gss_recoded codes that match LAD_2023: 318/318 (100.0%)

🔍 CONCLUSION: df_in_gss_recoded appears to use LAD_2023 codes
   Evidence: Contains 4 codes that only exist in 2023
Unique codes in df_in_gss_recoded gss_in: 318

Codes in df_in_gss_recoded that are ONLY in LAD_2021 (0):

Codes in df_in_gss_recoded that are ONLY in LAD_2023 (4):
  E06000063
  E06000064
  E06000065
  E06000066

OVERLAP ANALYSIS:
df_in_gss_recoded codes that match LAD_2021: 314/318 (98.7%)
df_in_gss_recoded codes that match LAD_2023: 318/318 (100.0%)

🔍 CONCLUSION: df_in_gss_recoded appears to use LAD_2023 codes
   Evidence: Contains 4 codes that only exist in 2023

📊 COMPARISON WITH 